# BODAQS Macro Analysis Dashboard

This notebook drives the extracted backend (`bodaqs_analysis`) with minimal editing.

Workflow:
1. Load CSV
2. Normalize (zero + scale)
3. Estimate velocity/acceleration
4. Load event schema + detect events
5. Inspect metrics / quick plots


In [1]:
import pandas as pd
import numpy as np

from bodaqs_analysis import run_macro
from bodaqs_analysis import normalize_and_scale
from bodaqs_analysis import estimate_va_from_zeroed
from bodaqs_analysis import load_event_schema
from bodaqs_analysis import detect_events_from_schema
from bodaqs_analysis import extract_metrics_df





In [2]:
# ---- Inputs ----
CSV_PATH = r"C:\Users\Ben Connor\OneDrive\Documents\logs\Simple tests\2025-12-21_11-57-23.CSV"   # change to your log
SCHEMA_PATH = r"Jupyter Notebook\event_schema.yaml"

# Columns and full-ranges (mm) for normalization (edit as needed)
NORMALIZE_RANGES = {
    "front_shock [mm]": 170.0,
    "rear_shock [mm]": 165.0,
}

SAMPLE_RATE_HZ = 200   # if known; used for VA estimation when time col is absent


In [3]:
# Run the macro pipeline
results = run_macro(
    CSV_PATH,
    SCHEMA_PATH,
    normalize_ranges=NORMALIZE_RANGES,
    sample_rate_hz=SAMPLE_RATE_HZ,
)
session = results['session']
data = session['df']
events_df = results['events']
metrics_df = results['metrics']

data.head()


[DEBUG] deep_compression(rear_shock): raw candidates=668
[DEBUG] deep_compression(rear_shock): after debounce=253 (gap=0.3s, prefer_key=disp, prefer_abs=False)
[DEBUG] deep_compression(rear_shock): passed conditions=0
[DEBUG] deep_compression(front_shock): raw candidates=657
[DEBUG] deep_compression(front_shock): after debounce=228 (gap=0.3s, prefer_key=disp, prefer_abs=False)
[DEBUG] deep_compression(front_shock): passed conditions=0
[DEBUG] rebounds(rear_shock): raw candidates=921
[DEBUG] rebounds(rear_shock): after debounce=197 (gap=0.3s, prefer_key=disp, prefer_abs=False)
[DEBUG] rebounds(rear_shock): passed conditions=100
[DEBUG] rebounds(front_shock): raw candidates=1112
[DEBUG] rebounds(front_shock): after debounce=156 (gap=0.3s, prefer_key=disp, prefer_abs=False)
[DEBUG] rebounds(front_shock): passed conditions=111
[Detect] Built EVENTS_DF with 211 rows from 2 schema event(s) → 4 sensor-expanded event(s).
event_id  t0_time  start_index  end_index
rebounds    10.50         1030 

,timestamp,sample_id,front_shock [mm],front_shock_raw [counts],rear_shock [mm],rear_shock_raw [counts],mark,sd_busy,time_s,front_shock [mm]_norm,rear_shock [mm]_norm,front_shock [mm]_vel,front_shock [mm]_acc,rear_shock [mm]_vel,rear_shock [mm]_acc
0,03:57:23.028,29445.0,16.523832,677.0,24.054088,634.0,0.0,0.0,0.00,0.097199,0.145782,-26.973444,2477.581259,-133.413157,8130.771888
1,03:57:23.038,29446.0,16.325348,673.0,23.554031,626.0,0.0,0.0,0.01,0.096031,0.142752,-15.747990,2012.600559,-95.013101,7229.250443
2,03:57:23.048,29447.0,16.325348,673.0,23.302669,622.0,0.0,0.0,0.02,0.096031,0.141228,-6.847438,1547.619860,-61.120653,6327.728998
3,03:57:23.058,29448.0,16.325348,673.0,22.988464,617.0,0.0,1.0,0.03,0.096031,0.139324,-0.271791,1082.639161,-31.735811,5426.207552
4,03:57:23.068,29449.0,16.325348,673.0,22.674259,612.0,0.0,0.0,0.04,0.096031,0.137420,3.978953,617.658462,-6.858577,4524.686107


In [4]:
data2, norm_meta = normalize_and_scale(
    data,
    NORMALIZE_RANGES,
    zeroing_enabled=True,
    zero_window_s=1.0,
    clip_0_1=False,
    return_meta=True,
)

pd.DataFrame(norm_meta["per_column"])


,column,status,full_range,clip_0_1,zeroing,meta
0,front_shock [mm],ok,170.0,False,"{'enabled': True, 'per_segment': False, 'metho...","{'method': 'min_window_avg', 'offset': 0.0, 't..."
1,rear_shock [mm],ok,165.0,False,"{'enabled': True, 'per_segment': False, 'metho...","{'method': 'min_window_avg', 'offset': 0.0, 't..."


In [5]:
# Compute VA on base columns (zeroing is now in-place, so no *_zeroed columns)
va_cols = [c for c in NORMALIZE_RANGES.keys() if c in data2.columns]

va_df, va_meta = estimate_va_from_zeroed(
    data2,
    cols=va_cols,
    sample_rate_hz=SAMPLE_RATE_HZ,
    time_col="time_s",
    window_points=31,
    poly_order=3,
    vel_suffix="_vel",
    acc_suffix="_acc",
    strip_zeroed_suffix=False,   # nothing ends with _zeroed anymore
    return_meta=True,            # <-- key change
)

va_df.filter(regex="(^time_s$|time|_vel$|_acc$)").head()



,timestamp,time_s,front_shock [mm]_vel,front_shock [mm]_acc,rear_shock [mm]_vel,rear_shock [mm]_acc
0,03:57:23.028,0.00,-4.004906,113.594337,-54.919438,3637.151598
1,03:57:23.038,0.01,-3.453065,107.142033,-37.385265,3376.517630
2,03:57:23.048,0.02,-2.933486,100.689728,-21.154262,3115.883663
3,03:57:23.058,0.03,-2.446168,94.237424,-6.226429,2855.249695
4,03:57:23.068,0.04,-1.991112,87.785120,7.398235,2594.615727


In [6]:
# Load schema (opt into meta if you want the hash/provenance)
schema, schema_meta = load_event_schema(SCHEMA_PATH, return_meta=True)
schema_hash = schema_meta.get("sha256")

events_df = detect_events_from_schema(va_df, schema)
metrics_df = extract_metrics_df(events_df)

events_df.head(), metrics_df.head()


[DEBUG] deep_compression(rear_shock): raw candidates=668
[DEBUG] deep_compression(rear_shock): after debounce=253 (gap=0.3s, prefer_key=disp, prefer_abs=False)
[DEBUG] deep_compression(rear_shock): passed conditions=0
[DEBUG] deep_compression(front_shock): raw candidates=657
[DEBUG] deep_compression(front_shock): after debounce=228 (gap=0.3s, prefer_key=disp, prefer_abs=False)
[DEBUG] deep_compression(front_shock): passed conditions=0
[DEBUG] rebounds(rear_shock): raw candidates=501
[DEBUG] rebounds(rear_shock): after debounce=331 (gap=0.3s, prefer_key=disp, prefer_abs=False)
[DEBUG] rebounds(rear_shock): passed conditions=168
[DEBUG] rebounds(front_shock): raw candidates=552
[DEBUG] rebounds(front_shock): after debounce=299 (gap=0.3s, prefer_key=disp, prefer_abs=False)
[DEBUG] rebounds(front_shock): passed conditions=190
[Detect] Built EVENTS_DF with 358 rows from 2 schema event(s) → 4 sensor-expanded event(s).
event_id  t0_time  start_index  end_index
rebounds    10.50         1030  

(   event_id                 event_type       sensor  t0_index  t0_time  \
 0  rebounds  simple_threshold_crossing   rear_shock      1050    10.50   
 1  rebounds  simple_threshold_crossing   rear_shock      1430    14.30   
 2  rebounds  simple_threshold_crossing  front_shock      1431    14.31   
 3  rebounds  simple_threshold_crossing   rear_shock      1482    14.82   
 4  rebounds  simple_threshold_crossing  front_shock      1487    14.87   
 
    win_pre_s  win_post_s  start_index  end_index  edge_clip  ...    acc_at_t0  \
 0        0.2         0.8         1030       1131      False  ... -3616.864651   
 1        0.2         0.8         1410       1511      False  ... -7910.769662   
 2        0.2         0.8         1411       1512      False  ... -8908.393377   
 3        0.2         0.8         1462       1563      False  ... -6250.109085   
 4        0.2         0.8         1467       1568      False  ... -5017.358169   
 
    disp_norm_at_t0 trig_rebound_end_t0_index  trig_re

In [7]:
import matplotlib.pyplot as plt
import pandas as pd

col = next((c for c in metrics_df.columns if c.lower().endswith("peak_vel") or "peak" in c.lower()), None)
if col is None:
    print("No obvious peak metric column found. Available:", list(metrics_df.columns)[:25])
else:
    s = pd.to_numeric(metrics_df[col], errors="coerce").dropna()
    if s.empty:
        print(f"Column '{col}' is present but has no numeric values.")
    else:
        s.plot(kind="hist", bins=40)
        plt.title(col)
        plt.show()



No obvious peak metric column found. Available: ['event_id', 'event_type', 'sensor', 't0_time', 't0_index', 'start_index', 'end_index', 'm_int_vel_mean', 'm_int_vel_max', 'm_int_vel_min', 'm_int_vel_t_start', 'm_int_vel_t_end', 'm_int_disp_delta']
